### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="polish_companies_bankruptcy",
    dataset_year="2010",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5F600",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/polish_companies_bankruptcy/ && wget -P local-data-warehouse/polish_companies_bankruptcy/ https://archive.ics.uci.edu/static/public/365/polish+companies+bankruptcy+data.zip && unzip local-data-warehouse/polish_companies_bankruptcy/polish+companies+bankruptcy+data.zip -d local-data-warehouse/polish_companies_bankruptcy/ && rm local-data-warehouse/polish_companies_bankruptcy/polish+companies+bankruptcy+data.zip
""",
    # References
    academic_reference_bibtex="""@article{zikeba2016ensemble,
  title={Ensemble boosted trees with synthetic features generation in application to bankruptcy prediction},
  author={Zi{\k{e}}ba, Maciej and Tomczak, Sebastian K and Tomczak, Jakub M},
  journal={Expert systems with applications},
  volume={58},
  pages={93--101},
  year={2016},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="zikeba2016ensemble",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We only use data from year 5 (5year.arff), because it is the newest data and the target is bankruptcy status after only 1 year.
- We created semantically meaningful feature names.
- We removed duplicates.
- Anomaly: the data contains a lot of features created by feature engineering.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="company_bankrupt",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="company_bankrupt",
)

## Preprocessing

In [2]:
import arff
import pandas as pd

with open(dataset_mold.path /"5year.arff") as f:
        data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

target_feature = task_mold.target_column_name
df.columns = [
    "net_profit_to_total_assets",
    "total_liabilities_to_total_assets",
    "working_capital_to_total_assets",
    "current_assets_to_short_term_liabilities",
    "liquidity_days_ratio",
    "retained_earnings_to_total_assets",
    "ebit_to_total_assets",
    "book_value_equity_to_total_liabilities",
    "sales_to_total_assets",
    "equity_to_total_assets",
    "extended_profit_to_total_assets",
    "gross_profit_to_short_term_liabilities",
    "gross_profit_plus_depreciation_to_sales",
    "gross_profit_plus_interest_to_total_assets",
    "liabilities_days_ratio",
    "gross_profit_plus_depreciation_to_total_liabilities",
    "total_assets_to_total_liabilities",
    "gross_profit_to_total_assets",
    "gross_profit_to_sales",
    "inventory_days_ratio",
    "sales_growth_ratio",
    "operating_profit_to_total_assets",
    "net_profit_to_sales",
    "three_year_gross_profit_to_total_assets",
    "equity_minus_share_capital_to_total_assets",
    "net_profit_plus_depreciation_to_total_liabilities",
    "operating_profit_to_financial_expenses",
    "working_capital_to_fixed_assets",
    "log_total_assets",
    "net_liabilities_to_sales",
    "gross_profit_plus_interest_to_sales",
    "current_liabilities_days_ratio",
    "operating_expenses_to_short_term_liabilities",
    "operating_expenses_to_total_liabilities",
    "sales_profit_to_total_assets",
    "total_sales_to_total_assets",
    "current_assets_minus_inventories_to_long_term_liabilities",
    "constant_capital_to_total_assets",
    "sales_profit_to_sales",
    "liquid_assets_to_short_term_liabilities",
    "liabilities_to_adjusted_operating_profit",
    "operating_profit_to_sales",
    "receivables_plus_inventory_turnover_days",
    "receivables_days_ratio",
    "net_profit_to_inventory",
    "current_assets_minus_inventory_to_short_term_liabilities",
    "inventory_days_cost_ratio",
    "ebitda_to_total_assets",
    "ebitda_to_sales",
    "current_assets_to_total_liabilities",
    "short_term_liabilities_to_total_assets",
    "short_term_liabilities_days_cost_ratio",
    "equity_to_fixed_assets",
    "constant_capital_to_fixed_assets",
    "working_capital_absolute",
    "gross_margin",
    "adjusted_liquidity_ratio",
    "total_costs_to_total_sales",
    "long_term_liabilities_to_equity",
    "inventory_turnover_ratio",
    "receivables_turnover_ratio",
    "short_term_liabilities_days_ratio",
    "sales_to_short_term_liabilities",
    "sales_to_fixed_assets",
    target_feature,
]

df[target_feature] = df[target_feature].map({"1": "Yes", "0": "No"}).astype("category")

# conflicting duplicates drop (without target column)
df = df.drop_duplicates(subset=[c for c in df.columns if c != task_mold.target_column_name], keep=False)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 5,790
Columns: 65
Use sampling: False (sample size: 5,790)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['liquidity_days_ratio', 'working_capital_absolute', 'liquid_assets_to_short_term_liabilities', 'gross_profit_plus_interest_to_sales', 'gross_profit_plus_depreciation_to_total_liabilities', 'liabilities_days_ratio', 'operating_expenses_to_total_liabilities', 'gross_profit_to_short_term_liabilities', 'gross_profit_to_sales', 'net_profit_plus_depreciation_to_total_liabilities']
Rows remaining as candidates after top-10 filter: 10 (of 5,790)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,net_profit_to_total_assets,total_liabilities_to_total_assets,working_capital_to_total_assets,current_assets_to_short_term_liabilities,liquidity_days_ratio,retained_earnings_to_total_assets,ebit_to_total_assets,book_value_equity_to_total_liabilities,sales_to_total_assets,equity_to_total_assets,extended_profit_to_total_assets,gross_profit_to_short_term_liabilities,gross_profit_plus_depreciation_to_sales,gross_profit_plus_interest_to_total_assets,liabilities_days_ratio,gross_profit_plus_depreciation_to_total_liabilities,total_assets_to_total_liabilities,gross_profit_to_total_assets,gross_profit_to_sales,inventory_days_ratio,sales_growth_ratio,operating_profit_to_total_assets,net_profit_to_sales,three_year_gross_profit_to_total_assets,equity_minus_share_capital_to_total_assets,net_profit_plus_depreciation_to_total_liabilities,operating_profit_to_financial_expenses,working_capital_to_fixed_assets,log_total_assets,net_liabilities_to_sales,gross_profit_plus_interest_to_sales,current_liabilities_days_ratio,operating_expenses_to_short_term_liabilities,operating_expenses_to_total_liabilities,sales_profit_to_total_assets,total_sales_to_total_assets,current_assets_minus_inventories_to_long_term_liabilities,constant_capital_to_total_assets,sales_profit_to_sales,liquid_assets_to_short_term_liabilities,liabilities_to_adjusted_operating_profit,operating_profit_to_sales,receivables_plus_inventory_turnover_days,receivables_days_ratio,net_profit_to_inventory,current_assets_minus_inventory_to_short_term_liabilities,inventory_days_cost_ratio,ebitda_to_total_assets,ebitda_to_sales,current_assets_to_total_liabilities,short_term_liabilities_to_total_assets,short_term_liabilities_days_cost_ratio,equity_to_fixed_assets,constant_capital_to_fixed_assets,working_capital_absolute,gross_margin,adjusted_liquidity_ratio,total_costs_to_total_sales,long_term_liabilities_to_equity,inventory_turnover_ratio,receivables_turnover_ratio,short_term_liabilities_days_ratio,sales_to_short_term_liabilities,sales_to_fixed_assets,company_bankrupt
0,-0.024622,0.69575,0.23866,1.3620,6.9025,-0.03091,-0.024622,0.43729,3.9185,0.30425,0.006002,-0.037346,0.002942,-0.024622,22025.00,0.016572,1.4373,-0.024622,-0.006284,14.876,0.99532,-0.001915,-0.006284,0.053616,0.25164,0.016572,-0.062521,2.33870,3.6769,0.171910,-0.002653,61.059,5.9778,5.66450,-0.022609,3.91850,101.0100,0.31156,-0.005770,0.039764,0.677380,-0.000489,81.201,66.325,-0.154170,1.11980,14.791,-0.038066,-0.009715,1.2906,0.65929,0.16729,2.9814,3.0530,1134.10,-0.005770,-0.080927,1.006200,0.024023,24.5360,5.5032,61.412,5.9435,38.3980,No
1,0.370150,0.37631,0.39923,2.0609,36.2850,0.00000,0.481240,1.65740,3.8763,0.62369,0.609720,1.278800,0.133850,0.481240,264.73,1.378800,2.6574,0.481240,0.124150,0.000,1.17570,0.533700,0.095491,1.408600,0.62065,1.083600,4.153900,1.77870,3.2170,-0.005092,0.124220,41.740,8.7446,8.74460,0.585560,3.87630,NaN,0.62369,0.151060,1.242000,0.021956,0.137680,29.016,29.016,NaN,2.06090,0.000,0.496080,0.127980,2.0609,0.37631,0.11436,2.7787,2.7787,658.04,0.151060,0.593490,0.878260,0.000000,NaN,12.5790,35.435,10.3010,17.2700,No
2,0.009359,0.46223,0.11333,1.2549,-92.8570,0.00000,0.009478,1.16340,1.4469,0.53777,0.028016,0.021318,0.036509,0.009478,3193.80,0.114280,2.1634,0.009478,0.006550,116.640,0.89136,0.027737,0.006469,-0.161410,0.49595,0.114030,1.496200,0.25637,3.8558,0.316610,0.015220,114.010,3.2014,3.07930,0.023555,1.44690,5.5037,0.55513,0.016280,0.024769,0.216750,0.019170,137.970,21.323,0.020241,0.21489,118.570,-0.015611,-0.010789,1.2071,0.44460,0.31236,1.2165,1.2558,813.11,0.016280,0.017404,0.993480,0.032281,3.1292,17.1170,112.160,3.2544,3.2731,No
3,0.085659,0.28314,0.48309,2.7062,-4.5713,0.18313,0.107660,2.12670,1.0215,0.60217,0.107660,0.380230,0.059843,0.107660,641.01,0.569410,3.5318,0.107660,0.039960,43.494,1.22280,0.094664,0.031794,0.230380,0.60217,0.491710,0.358930,2.06660,4.6094,0.093601,0.039960,39.185,9.3148,0.33433,0.094664,2.75720,NaN,0.60217,0.035137,0.797590,0.062799,0.035137,73.213,29.719,0.26

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,company_bankrupt,category,0.0,0.00,2.0,"No, Yes"
1,current_assets_minus_inventories_to_long_term_liabilities,float64,2502.0,43.21,3225.0,"2.7727, 4.1023, 1.0662, 1.3454, 11.675, 1.534, 18.721, 1.8035, 1.1776, 5.2905"
2,operating_profit_to_financial_expenses,float64,389.0,6.72,4935.0,"0.0, 1.397, 2.6295, 1.3454, 7.0649, 0.7183, 3.0986, 1.2127, 1.1694, 7.1814"
3,net_profit_to_inventory,float64,264.0,4.56,5367.0,"0.0, 0.132, 0.161, 1.5193, 2.907, 1.668, 0.368, 1.0742, 0.3118, 0.1546"
4,inventory_turnover_ratio,float64,264.0,4.56,5319.0,"11.045, 11.257, 12.364, 10.34, 5.2512, 17.114, 12.16, 9.2509, 4.0762, 7.4142"
5,three_year_gross_profit_to_total_assets,float64,135.0,2.33,5496.0,"0.0, 0.1345, 1.4774, 0.735, 0.1304, 0.2073, 0.1976, 0.7013, 0.2343, 0.0423"
6,working_capital_to_fixed_assets,float64,105.0,1.81,5555.0,"1.2599, 1.0404, 2.5396, 1.5311, 2.5157, 0.5859, 1.0304, 13.99, 2.0996, 6.7331"
7,equity_to_fixed_assets,float64,105.0,1.81,5415.0,"2.2499, 1.0391, 1.3602, 1.2216, 1.8225, 1.5962, 1.0107, 1.3816, 2.3016, 1.2624"
8,constant_capital_to_fixed_assets,float64,105.0,1.81,5298.0,"1.0371, 2.2499, 1.1296, 1.6298, 1.3008, 1.9589, 1.56, 1.0736, 1.2271, 1.1915"
9,sales_to_fixed_assets,float64,105.0,1.81,5493.0,"1.6959, 6.7586, 2.682, 1.8704, 16.589, 3.5166, 14.521, 6.0223, 2.9985, 0.6372"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
net_profit_to_total_assets,5787.0,-0.024261,6.227187,-4.638900e+02,8.745900e+01
total_liabilities_to_total_assets,5787.0,0.466893,5.810483,-4.308700e+02,7.241600e+01
working_capital_to_total_assets,5787.0,0.187873,1.189212,-7.206700e+01,2.833600e+01
current_assets_to_short_term_liabilities,5769.0,4.908487,92.370898,-4.031100e-01,6.845800e+03
liquidity_days_ratio,5779.0,19.568831,21751.730364,-1.076400e+06,1.250100e+06
retained_earnings_to_total_assets,5787.0,0.021568,10.095067,-4.638900e+02,5.432500e+02
ebit_to_total_assets,5787.0,-0.115911,9.150502,-5.174800e+02,5.530000e+00
book_value_equity_to_total_liabilities,5772.0,5.739466,103.400811,-3.735100e+00,6.868500e+03
sales_to_total_assets,5789.0,1.596409,1.561708,-3.496000e+00,6.560700e+01
equity_to_total_assets,5787.0,0.544601,5.823079,-7.144400e+01,3.398500e+02


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column           rank                    
company_bankrupt 1       No   5384  92.99
                 2      Yes    406   7.01

In [8]:
# Target Distribution
target_df

,count,pct
company_bankrupt,,
No,5384,92.99
Yes,406,7.01


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to polish_companies_bankruptcy/019d9d2a-a55b-7465-b41d-7a7c3cc16d13
019d9d2a-a55b-7465-b41d-7a7c3cc16d13
891c33a6391bea681c6bfb6db2a5b8eeb1ea42b9534f085dba71ca58db521300
